Họ và tên: Nguyễn Đình Khanh

MSSV: 24110244

Link Github: https://github.com/qilskcter/TriTueNhanTao

# PEAS CỦA MÁY HÚT BỤI

P (Performance): Độ sạch của sàn, lượng pin tiêu thụ, thời gian hoàn thành.

E (Environment): Sàn nhà, thảm, chướng ngại vật (chân bàn, ghế).

A (Actuators): Bánh xe để di chuyển, chổi quét, động cơ hút bụi.

S (Sensors): Cảm biến va chạm (Bumper), cảm biến hồng ngoại phát hiện bụi, cảm biến vách (chống rơi cầu thang).

### MÁY HÚT BỤI

In [19]:
import random

ACTION_SUCK = "SUCK"
MOVE_UP = "UP"
MOVE_DOWN = "DOWN"
MOVE_LEFT = "LEFT"
MOVE_RIGHT = "RIGHT"

DIRTY = "DIRTY"
CLEAN = "CLEAN"

DIRTY_VAL, CLEAN_VAL = 1, 0

In [20]:
def interpret_input(percept):
    return {
        'location': percept['location'],
        'status': percept['status']
    }

In [21]:
def rule_match(state, max_x, max_y):
    if state['status'] == DIRTY:
        return ACTION_SUCK
    
    x, y = state['location']
    possible_moves = [MOVE_UP, MOVE_DOWN, MOVE_LEFT, MOVE_RIGHT]
    
    if x == 0: 
        if MOVE_LEFT in possible_moves: 
            possible_moves.remove(MOVE_LEFT)
    if x == max_x: 
        if MOVE_RIGHT in possible_moves: 
            possible_moves.remove(MOVE_RIGHT)
    if y == 0: 
        if MOVE_UP in possible_moves: 
            possible_moves.remove(MOVE_UP)
    if y == max_y: 
        if MOVE_DOWN in possible_moves: 
            possible_moves.remove(MOVE_DOWN)
        
    return random.choice(possible_moves)

In [22]:
def simple_reflex_agent(percept, max_x, max_y):
    state = interpret_input(percept)
    
    action = rule_match(state, max_x, max_y)
    
    return action

In [23]:
MAX_X = 2   # Kích thước sàn nhà
MAX_Y = 2

percept_1 = {'location': (0, 0), 'status': DIRTY}
print(f"Bước 1: Robot tại (0,0), ô Bẩn -> Hành động: {simple_reflex_agent(percept_1, MAX_X, MAX_Y)}")

percept_2 = {'location': (2, 2), 'status': CLEAN}
print(f"Bước 2: Robot tại (2,2), ô Sạch -> Hành động: {simple_reflex_agent(percept_2, MAX_X, MAX_Y)}")

percept_3 = {'location': (1, 1), 'status': DIRTY}
print(f"Bước 3: Robot tại (1,1), ô Bẩn -> Hành động: {simple_reflex_agent(percept_3, MAX_X, MAX_Y)}")

Bước 1: Robot tại (0,0), ô Bẩn -> Hành động: SUCK
Bước 2: Robot tại (2,2), ô Sạch -> Hành động: UP
Bước 3: Robot tại (1,1), ô Bẩn -> Hành động: SUCK


In [24]:
def interpret_input_from_matrix(matrix, location):
    x, y = location
    status = DIRTY if matrix[x, y] == DIRTY_VAL else CLEAN
    return {'location': location, 'status': status}

In [25]:
def rule_match(state, max_x, max_y):
    """Quyết định hành động dựa trên các quy tắc phản xạ đơn giản"""
    # Quy tắc ưu tiên: Nếu bẩn thì phải hút ngay
    if state['status'] == DIRTY:
        return ACTION_SUCK
    
    x, y = state['location']
    possible_moves = [MOVE_UP, MOVE_DOWN, MOVE_LEFT, MOVE_RIGHT]
    
    # Logic chặn biên để tránh lỗi IndexError (out of bounds)
    if x == 0: 
        if MOVE_UP in possible_moves: possible_moves.remove(MOVE_UP)
    if x == max_x: 
        if MOVE_DOWN in possible_moves: possible_moves.remove(MOVE_DOWN)
    if y == 0: 
        if MOVE_LEFT in possible_moves: possible_moves.remove(MOVE_LEFT)
    if y == max_y: 
        if MOVE_RIGHT in possible_moves: possible_moves.remove(MOVE_RIGHT)
        
    return random.choice(possible_moves)

In [26]:
def simple_reflex_agent(percept, max_x, max_y):
    """Hàm Agent chính điều phối việc nhận cảm và ra quyết định"""
    state = interpret_input_from_matrix(percept['matrix'], percept['location'])
    return rule_match(state, max_x, max_y)

In [27]:
import numpy as np

DIRTY_VAL = 1
CLEAN_VAL = 0

def create_random_floor(rows, cols, dirty_prob=0.4):
    return np.random.choice([CLEAN_VAL, DIRTY_VAL], size=(rows, cols), p=[1-dirty_prob, dirty_prob])

rows, cols = 3, 3
floor_matrix = create_random_floor(rows, cols)
# floor_matrix = np.array([[DIRTY_VAL, DIRTY_VAL, DIRTY_VAL],
#                          [DIRTY_VAL, DIRTY_VAL, DIRTY_VAL],
#                          [DIRTY_VAL, DIRTY_VAL, DIRTY_VAL]])
print("Ma trận sàn nhà mục tiêu (1 là bẩn):\n", floor_matrix)

current_pos = [0, 0]
max_steps = 50
step_count = 0

max_row_idx = rows - 1
max_col_idx = cols - 1

print(f"\nBắt đầu mô phỏng tại vị trí: {current_pos}\n")

while np.any(floor_matrix == DIRTY_VAL) and step_count < max_steps:
    percept = {
        'matrix': floor_matrix, 
        'location': tuple(current_pos)
    }
    
    action = simple_reflex_agent(percept, max_row_idx, max_col_idx)
    
    print(f"Bước {step_count + 1}: Tại {current_pos} -> Quyết định: {action}")
    
    if action == ACTION_SUCK:
        floor_matrix[current_pos[0], current_pos[1]] = CLEAN_VAL
    elif action == MOVE_UP:
        current_pos[0] -= 1
    elif action == MOVE_DOWN:
        current_pos[0] += 1
    elif action == MOVE_LEFT:
        current_pos[1] -= 1
    elif action == MOVE_RIGHT:
        current_pos[1] += 1
    
    step_count += 1

print("\n--- KẾT QUẢ SAU KHI CHẠY ---")
print("Ma trận hiện tại:")
print(floor_matrix)

if not np.any(floor_matrix == DIRTY_VAL):
    print(f"Thành công! Sàn sạch sau {step_count} bước.")
else:
    print(f"Dừng lại: Robot chưa dọn xong sau {max_steps} bước.")

Ma trận sàn nhà mục tiêu (1 là bẩn):
 [[1 0 0]
 [0 0 0]
 [1 0 1]]

Bắt đầu mô phỏng tại vị trí: [0, 0]

Bước 1: Tại [0, 0] -> Quyết định: SUCK
Bước 2: Tại [0, 0] -> Quyết định: RIGHT
Bước 3: Tại [0, 1] -> Quyết định: RIGHT
Bước 4: Tại [0, 2] -> Quyết định: DOWN
Bước 5: Tại [1, 2] -> Quyết định: LEFT
Bước 6: Tại [1, 1] -> Quyết định: LEFT
Bước 7: Tại [1, 0] -> Quyết định: DOWN
Bước 8: Tại [2, 0] -> Quyết định: SUCK
Bước 9: Tại [2, 0] -> Quyết định: RIGHT
Bước 10: Tại [2, 1] -> Quyết định: LEFT
Bước 11: Tại [2, 0] -> Quyết định: UP
Bước 12: Tại [1, 0] -> Quyết định: RIGHT
Bước 13: Tại [1, 1] -> Quyết định: DOWN
Bước 14: Tại [2, 1] -> Quyết định: LEFT
Bước 15: Tại [2, 0] -> Quyết định: RIGHT
Bước 16: Tại [2, 1] -> Quyết định: RIGHT
Bước 17: Tại [2, 2] -> Quyết định: SUCK

--- KẾT QUẢ SAU KHI CHẠY ---
Ma trận hiện tại:
[[0 0 0]
 [0 0 0]
 [0 0 0]]
Thành công! Sàn sạch sau 17 bước.
